# Stabilizer Approach Experiment

This experiment aims to explore the difference in fidelity between different approaches for quantum error correction.

## Tasks:

- [x] Create routine to initialize states, gates, and trotterizer
- Create independent variable test routine for qutrits
    - [ ] Lowering gate into 2 qubit ancilla space, then raising back up
    - [ ] Performing ancilla check using mapping directly |f> -> |1>, |g> -> |0>
    - [ ] Check performing repetition first then phase or phase then repetition
    - [ ] Create routine to analyze fidelity with different independent variables


In [4]:
# Import required classes
import os
from typing import Callable

import matplotlib.pyplot as plt
import numpy as np
import qutip as qt

from quantum_logical.channel import AmplitudeDamping, PhaseDamping
from quantum_logical.trotter import TrotterGroup


In [5]:
# Generate Trotterizer with Amplitude and Phase Damping Channels
def generate_trotterizer(trotter_dt, T1, T2, dim, num_qubits):
    """Generate a TrotterGroup with amplitude and phase damping channels.

    Args:
        trotter_dt (float): Time step for the Trotterization.
        T1 (float): Relaxation time for amplitude damping.
        T2 (float): Dephasing time for phase damping.
        dim (int): Dimension of the system.
        num_qubits (int): Number of qubits in the system.

    Returns:
        TrotterGroup: Configured TrotterGroup with specified channels.
    """

    amp_damp = AmplitudeDamping(T1=T1, hilbert_space_dim=dim, num_qubits=num_qubits)
    phase_damp = PhaseDamping(T1=T1, T2=T2, hilbert_space_dim=dim, num_qubits=num_qubits)
    
    trotterizer = TrotterGroup(
        [amp_damp, phase_damp],
        trotter_dt,
    )
    
    return trotterizer


In [6]:
# Generate logical gates
def hadamard_operator(dim:int) -> qt.Qobj:
    """Create a logical Hadamard operator matrix for a given dimension.
    Args:
        dim (int): Dimension of the system. Must be between 1 and 4.
    Returns:
        qt.Qobj: Hadamard operator matrix.
    """
    match dim:
        case 2:
            return 1/np.sqrt(2) * qt.Qobj([[1, 1], [1, -1]])
        case 3:
            return qt.Qobj(
                [
                    [1/np.sqrt(2), 0, 1/np.sqrt(2)], 
                    [0, 1, 0], 
                    [1/np.sqrt(2), 0, -1/np.sqrt(2)]
                ])
        case _:
            raise ValueError("Dimension must be 2 or 3.")

def lowering_operator(dim:int, n:int) -> qt.Qobj:
    """Create a logical lowering operator matrix for a given dimension.
    Takes state |n> to |n-1 (mod d)>, and |n-1 (mod d)> to |n>.

    Args:
        dim (int): Dimension of the system. Must be between 1 and 4.
        n (int): The state to lower.
    Returns:
        qt.Qobj: Lowering operator matrix.
    """

    # Check inputs
    if n <0 or n >= dim:
        raise ValueError("State n must be between 0 and dim-1.")
    
    if dim < 1 or dim > 4:
        raise ValueError("Dimension must be between 1 and 4.")
    
    # Generate indentity on all states, then modify target state
    gate = np.eye(dim,dtype=complex)
    target_index = (n - 1) % dim

    gate[target_index, n] = 1
    gate[n, n] = 0
    gate[n,target_index] = 1
    gate[target_index,target_index] = 0

    return qt.Qobj(gate)



z_gate:qt.Qobj = qt.Qobj([[1, 0, 0],[0, 1, 0], [0, 0, -1]])

In [7]:
def cnot_operator(num_qudits: int, dim: int, control_idx: int, target_idx: int, trigger:int) -> qt.Qobj:
    """
    Creates a CNOT gate where all particles are qudits of dimension 'dim'.
    
    Trigger: Control Qudit is in state |trigger>.
    Action:  Target Qudit swaps |0> <-> |trigger> (Subspace X), 
             leaving states |2>...|dim-1> unchanged.
    
    Args:
        num_qudits: Total number of qudits in the system.
        dim: The constant dimension for all qudits (must be >= 2).
        control_idx: Index of the control qudit.
        target_idx: Index of the target qudit.
    """
    
    # 1. Validation
    if dim < 2:
        raise ValueError("Dimension must be at least 2 to have a 0-1 subspace.")
    if control_idx < 0 or control_idx >= num_qudits:
        raise ValueError("Control index out of range.")
    if target_idx < 0 or target_idx >= num_qudits:
        raise ValueError("Target index out of range.")
    if control_idx == target_idx:
        raise ValueError("Control and target indices must be different.")

    # 2. Define the 'Subspace X' Operator for the target
    # Start with an empty matrix
    X_sub = qt.Qobj(np.zeros((dim, dim), dtype=complex))
    
    # Add the swap for the subspace: |0><1| + |1><0|
    X_sub += qt.basis(dim, 0) * qt.basis(dim, 1).dag()
    X_sub += qt.basis(dim, 1) * qt.basis(dim, 0).dag()
    
    # Add Identity for all higher states: |k><k| for k >= 2
    for k in range(2, dim):
        X_sub += qt.basis(dim, k) * qt.basis(dim, k).dag()

    # 3. Initialize total Unitary U
    # Total system dimension is dim^num_qudits
    # We define the shape dimensions list once: [dim, dim, ..., dim]
    sys_dims = [dim] * num_qudits
    total_dim = dim ** num_qudits
    
    U = qt.Qobj(np.zeros((total_dim, total_dim), dtype=complex), 
                dims=[sys_dims, sys_dims])

    # 4. Iterate over Control States
    for c_state in range(dim):
        
        # A. Determine Target Action
        if c_state == trigger:
            target_op = X_sub         # Swap |0> and |1>
        else:
            target_op = qt.identity(dim) # Do nothing
            
        # B. Create Control Projector: |c><c|
        P_c = qt.basis(dim, c_state) * qt.basis(dim, c_state).dag()
        
        # C. Build Component List
        op_list = []
        for i in range(num_qudits):
            if i == control_idx:
                op_list.append(P_c)
            elif i == target_idx:
                op_list.append(target_op)
            else:
                op_list.append(qt.identity(dim))
        
        # D. Add to Total U
        U += qt.tensor(op_list)

    return U

In [8]:
def erasure_check_operator(num_qudits: int, dim: int, control_idx: int, target_idx: int) -> qt.Qobj:
    """
    Creates a C2-X gate (Erasure Check) where all qudits have the same dimension.
    
    Logic:
      - If Control is in state |2>: Apply X (flip 0<->1) to Target.
      - If Control is in state |0>, |1>, or >|2>: Do nothing to Target (Identity).
    
    Args:
        num_qudits (int): Total number of qudits in the system.
        dim (int): The Hilbert space dimension of every qudit (must be >= 3).
        control_idx (int): Index of the qudit being checked for leakage.
        target_idx (int): Index of the ancilla used to flag the error.
        
    Returns:
        qt.Qobj: The tensor product operator for the full system.
    """
    
    # --- Input Validation ---
    if dim < 3:
        raise ValueError(f"Dimension must be at least 3 to check for state |2>. Received dim={dim}.")
    
    if control_idx >= num_qudits or target_idx >= num_qudits:
        raise ValueError(f"Indices ({control_idx}, {target_idx}) out of range for {num_qudits} qudits.")

    # --- Construct Operators ---
    
    # 1. Control Projectors
    # P_2: Projector onto the leakage state |2><2|
    P_2 = qt.basis(dim, 2) * qt.basis(dim, 2).dag()
    
    # P_not2: Projector onto everything else (Identity - P_2)
    P_not2 = qt.qeye(dim) - P_2

    # 2. Target Operators
    # Create a logical bit flip (0 <-> 1) for the generic dimension
    # Leaves states |2>, |3>... untouched
    X_op = qt.sigmax()

    I_op = qt.qeye(dim)

    # --- Build Tensor Product ---
    # We sum two terms:
    # Term A: (Control is NOT |2>) (Target is Identity)
    # Term B: (Control IS |2>)     (Target is X)

    ops_A = [] # List for Term A
    ops_B = [] # List for Term B

    for i in range(num_qudits):
        if i == control_idx:
            ops_A.append(P_not2)
            ops_B.append(P_2)
        elif i == target_idx:
            ops_A.append(I_op)
            ops_B.append(X_op)
        else:
            # Identity on all other qudits
            ops_A.append(I_op)
            ops_B.append(I_op)

    # Sum the terms to get the Controlled-Unitary
    return qt.tensor(ops_A) + qt.tensor(ops_B)

In [46]:
def run_unitary_circuit(trotterer:TrotterGroup, unitary_circuit:list[qt.Qobj], initial_state:qt.Qobj, gate_times:list[float])-> list[qt.Qobj]:
    """Trotterizes a given circuit with a starting state and given gate times.

    Args:
        trotterer (TrotterGroup): The totterizer to use.
        unitary_circuit (list[qt.Qobj]): The list of unitary gates in the circuit.
        initial_state (qt.Qobj): The initial state to start the circuit from.
        gate_times (list[float]): The time each gate is applied for.

    Returns:
        list[qt.Qobj]: The list of states for each time step.
    """
    state_list = [initial_state]
    running_time = 0.0
    for gate, gate_time in zip(unitary_circuit,gate_times):
        state_list.extend(trotterer.apply(state=state_list[-1], discrete_unitary=gate, duration=gate_time))
        running_time += gate_time
    
    return state_list


In [34]:

def plot_results(trotter_dt:float, state_list:list[qt.Qobj], output_path:str)->None:
    """Plot the results of a circuit run.

    Args:
        state_list (list[qt.Qobj]): The list of states to plot.
        output_path (str): The path to save the output plot.
    """
    if not os.path.exists(output_path):
        os.makedirs(output_path)


    time_steps = np.arange(0, trotter_dt * (len(state_list)), trotter_dt)

    probabilities = [state.ptrace([0,1,2]).diag().real for state in state_list]
    probabilities = np.array(probabilities)

    fidelities = [qt.fidelity(state.ptrace([0,1,2]), state_list[0].ptrace([0,1,2])) for state in state_list]
    fidelities = np.array(fidelities)

    plt.figure(figsize=(10,6))
    for i in [0, 13, 26]: #range(probabilities.shape[1]):
        plt.plot(time_steps, probabilities[:,i], label=f'State |{np.base_repr(i, base=3)}>')
    
    #plt.axvline(x=0, color='r', linestyle='--', label='Hadamard 1')
    #plt.axvline(x=.03, color='r', linestyle='--', label='Lowering 1')
    #plt.axvline(x=.53, color='r', linestyle='--', label='CNOT_1')
    #plt.axvline(x=1.03, color='r', linestyle='--', label='CNOT_2')
    #plt.axvline(x=1.53, color='r', linestyle='--', label='CNOT_3')
    #plt.axvline(x=2.03, color='r', linestyle='--', label='CNOT_4')
    #plt.axvline(x=2.06, color='r', linestyle='--', label='Lowering 2')
    #plt.axvline(x=2.09, color='r', linestyle='--', label='Hadamard 2')
    plt.xlabel('Time Step')
    plt.ylabel('Probability')
    plt.title('State Probabilities Over Time')
    plt.legend()
    plt.savefig(f"{output_path}/probabilities_plot.png")
    plt.show()


    plt.figure(figsize=(10,6))
    plt.plot(time_steps, fidelities, label='Fidelity')
    plt.xlabel('Time Step')
    plt.ylabel('Fidelities')
    plt.title('Fidelities Over Time')
    plt.legend()
    plt.savefig(f"{output_path}/probabilities_plot.png")
    plt.show()


In [ ]:
def experiment_run(trotterer:TrotterGroup, single_qudit_times:float, two_qudit_times:float, setup_circuit:Callable[[tuple[float,float]], tuple[list[qt.Qobj], list[float], list[qt.Qobj], list[qt.Qobj]]], output_path:str)->None:
    """Runs an experiement for a circuit defined in setup_circuit for given trotterer,
    and output the resulting plots to output_path.

    Args:
        trotterer (TrotterGroup): The trotterizer used for simulation
        single_qudit_times (float): The time it takes to perform single qudit operations
        two_qudit_times (float): The time it takes to perform two qudit operations
        setup_circuit (Callable[[Tuple[float,float]], tuple[list[list[qt.Qobj]], list[list[float]]]]): the function which takes 
        the single and multi qubit times creates the quantum circuit with associated times
        output_path (str): The output directory, created anew if not already existing
    """

    
    psi_0:qt.Qobj = qt.tensor(*[(qt.basis(3,0) + qt.basis(3,2))/np.sqrt(2)]*3, *[qt.basis(3,0)]*3)
    rho_0 = psi_0 * psi_0.dag()
    
    
    circuits, gate_times, measurements, recovery_ops, recovery_times = setup_circuit(single_qudit_times, two_qudit_times)
    state_list = run_unitary_circuit(trotterer, circuits, rho_0, gate_times)

    # building branches based on measurement 
    # Assume measurement is instantaneous for simplicity
    proj_results_after_measurement = [(state_list[-1] * proj).tr() for proj in measurements]
    proj_states_after_measurment = [(proj * state_list[-1] * proj.dag()) for proj in measurements]

    # corrects the branches after measurement 
    corrected_states = []
    for i in range(len(recovery_ops)):
        if proj_results_after_measurement[i] != 0 and recovery_times[i] != 0.0: # ensures no division by zero in trotterization
            corrected_states.append(run_unitary_circuit(trotterer, recovery_ops[i], proj_states_after_measurment[i], recovery_times[i])[-1])
        else:
            recovery_unitary = recovery_ops[i][0]
            for j in recovery_ops[1:]:
                recovery_unitary *= recovery_ops[i][j]
            corrected_states.append(qt.Qobj(recovery_unitary * proj_states_after_measurment[i] * recovery_unitary.dag()))
    
    #for i in range(len(proj_results_after_measurement)):
        #total_time += proj_results_after_measurement[i] * sum(recovery_times[i])

    # recombining branches post correction
    for j in range(len(proj_results_after_measurement)):
        print("result type:", type(proj_results_after_measurement[j]))
        print("corrected states type:", type(corrected_states[j]))
    repetition_corrected_state = (sum([proj_results_after_measurement[j] * corrected_states[j] 
                                       for j in range(len(proj_results_after_measurement))]) / 
                                       sum([proj_results_after_measurement[j] * corrected_states[j] 
                                       for j in range(len(proj_results_after_measurement))]).tr())


    state_list.extend(run_unitary_circuit(trotterer, [qt.tensor([qt.qeye(3)]*6)], repetition_corrected_state, [1]))

    plot_results(trotterer.trotter_dt, state_list, output_path)


In [36]:
def serial_phase_circuit(single_qudit_time:float, two_qudit_time:float) -> tuple[list[qt.Qobj], list[float], list[qt.Qobj], list[list[qt.Qobj]], list[list[float]]]:
    """Sets up a circuit which applies phase errors followed by a repetition code.
    The circuit lowers the qutrit states before entangling with the ancilla qubits.
    Cnots are applied in series for the phase check.

    NOTE: The hadamard gates do not trotterize well. A solution is to have the trotterization step be the same as the single qudit time.

    Args:
        single_qudit_time (float): Time taken for single qudit operations.
        two_qudit_time (float): Time taken for two qudit operations.

    Returns:
        The circuit, associated gate times, measurement operators, recovery operators, and recovery times.
    """
    circuit = []
    gate_time = []

    # Apply hadamard to all state qutrits
    circuit.append(qt.tensor(hadamard_operator(3), hadamard_operator(3), hadamard_operator(3), qt.qeye(3), qt.qeye(3)))
    gate_time.append(single_qudit_time)

    # Apply lowering operator to all state qutrits
    circuit.append(qt.tensor(lowering_operator(3,2), lowering_operator(3,2), lowering_operator(3,2), qt.qeye(3), qt.qeye(3)))
    gate_time.append(single_qudit_time)

    # Apply CNOTs for repetition code
    circuit.append(cnot_operator(5, 3, 0, 3, 1))
    circuit.append(cnot_operator(5, 3, 1, 3, 1))
    circuit.append(cnot_operator(5, 3, 1, 4, 1))
    circuit.append(cnot_operator(5, 3, 2, 4, 1))
    gate_time.extend([two_qudit_time]*4)

    # Apply lowering operator to all qutrits to revert
    circuit.append(qt.tensor(lowering_operator(3,2), lowering_operator(3,2), lowering_operator(3,2), qt.qeye(3), qt.qeye(3)))
    gate_time.append(single_qudit_time)
    
    # Apply hadamard to all qutrits to revert
    circuit.append(qt.tensor(hadamard_operator(3), hadamard_operator(3), hadamard_operator(3), qt.qeye(3), qt.qeye(3)))
    gate_time.append(single_qudit_time)

    #Measurement operators on ancillae
    phase_measurements = [
        qt.tensor(
            *[qt.qeye(3)]*3, 
            (qt.tensor(qt.basis(3, i), qt.basis(3, j)) * (qt.tensor(qt.basis(3, i), qt.basis(3, j))).dag())
        ) 
        for i in range(3) for j in range(3)
    ]

    # correction operators
    r00 = qt.tensor([qt.qeye(3)] * 5)
    r01 = qt.tensor(qt.qeye(3), qt.qeye(3), z_gate, *[qt.qeye(3)] * 2)
    r10 = qt.tensor(z_gate, *([qt.qeye(3)] * 2), *[qt.qeye(3)] * 2)
    r11 = qt.tensor(qt.qeye(3), z_gate, qt.qeye(3), *[qt.qeye(3)] * 2)


    r00 = r02 = r20 = r22 = r12 = r21 = qt.tensor([qt.qeye(3)] * 5)
    r01 = qt.tensor(qt.qeye(3), qt.qeye(3), z_gate, qt.tensor([qt.qeye(3)] * 2))
    r10 = qt.tensor(z_gate, qt.tensor([qt.qeye(3)] * 4))
    r11 = qt.tensor(qt.qeye(3), z_gate, qt.qeye(3), qt.tensor([qt.qeye(3)] * 2))
    recovery_ops = [[r00], [r01], [r02], [r10], [r11], [r12], [r20], [r21], [r22]]
    recovery_times = [[0], [single_qudit_time], [0], [single_qudit_time], [single_qudit_time], [0], [0], [0], [0]]

    return circuit, gate_time, phase_measurements, recovery_ops, recovery_times

In [58]:
def serial_erasure_circuit(single_qudit_time:float, two_qudit_time:float) -> tuple[list[qt.Qobj], list[float], list[qt.Qobj], list[qt.Qobj], list[float]]:
    """Sets up a circuit which applies erasure errors followed by a repetition code.
    The circuit lowers the qutrit states before entangling with the ancilla qubits.
    Cnots are applied in series for the erasure check.

    Args:
        single_qudit_time (float): Time taken for single qudit operations.
        two_qudit_time (float): Time taken for two qudit operations.

    Returns:
        tuple[list[qt.Qobj], list[float]]: The circuit, associated gate times, measurement operators, recovery operators, and recovery times.
    """
    circuit = []
    gate_time = []

    cnot1 = cnot_operator(num_qudits=6, dim=3, control_idx=0, target_idx=3, trigger=1)
    cnot2 = cnot_operator(num_qudits=6, dim=3, control_idx=1, target_idx=4, trigger=1)
    cnot3 = cnot_operator(num_qudits=6, dim=3, control_idx=2, target_idx=5, trigger=1)

    cnot4 = cnot_operator(num_qudits=6, dim=3, control_idx=0, target_idx=2, trigger=2)
    cnot5 = cnot_operator(num_qudits=6, dim=3, control_idx=0, target_idx=1, trigger=2)
    cnot6 = cnot_operator(num_qudits=6, dim=3, control_idx=1, target_idx=0, trigger=2)
    cnot7 = cnot_operator(num_qudits=6, dim=3, control_idx=1, target_idx=2, trigger=2)
    cnot8 = cnot_operator(num_qudits=6, dim=3, control_idx=2, target_idx=0, trigger=2)
    cnot9 = cnot_operator(num_qudits=6, dim=3, control_idx=2, target_idx=1, trigger=2)

    hadamard_layer = qt.tensor(hadamard_operator(3), hadamard_operator(3), hadamard_operator(3), qt.qeye(3), qt.qeye(3), qt.qeye(3))
    # Apply CNOTs for repetition code
    circuit.append(cnot1)
    circuit.append(cnot2)
    circuit.append(cnot3)
    gate_time.extend([two_qudit_time]*3)

    x_gate = qt.Qobj(np.array([[0,0,1],[0,1,0],[1,0,0]]))

    #Measurement operators on ancillae
    measurements = [qt.tensor(qt.qeye(3), qt.qeye(3), qt.qeye(3), qt.tensor(qt.basis(3, i), qt.basis(3, j), qt.basis(3, k))  
                   * qt.tensor(qt.basis(3, i), qt.basis(3, j), qt.basis(3, k)).dag()) 
             for i in [0,1] for j in [0,1] for k in [0,1]]
    
    # correction_operators
    r000 = r111 = [qt.tensor(*[qt.qeye(3)]*6)]
    r001 = [hadamard_layer, qt.tensor(qt.qeye(3), qt.qeye(3), x_gate, qt.qeye(3), qt.qeye(3), qt.qeye(3)), cnot4, hadamard_layer]
    r010 = [hadamard_layer, qt.tensor(qt.qeye(3), x_gate, qt.qeye(3), qt.qeye(3), qt.qeye(3), qt.qeye(3)), cnot5, hadamard_layer]
    r011 = [hadamard_layer, qt.tensor(qt.qeye(3), x_gate, qt.qeye(3), qt.qeye(3), qt.qeye(3), qt.qeye(3)), qt.tensor(qt.qeye(3), qt.qeye(3), x_gate, qt.qeye(3), qt.qeye(3), qt.qeye(3)), cnot5, cnot4, hadamard_layer]
    r100 = [hadamard_layer, qt.tensor(x_gate, qt.qeye(3), qt.qeye(3), qt.qeye(3), qt.qeye(3), qt.qeye(3)), cnot6, hadamard_layer]
    r101 = [hadamard_layer, qt.tensor(x_gate, qt.qeye(3), qt.qeye(3), qt.qeye(3), qt.qeye(3), qt.qeye(3)), qt.tensor(qt.qeye(3), qt.qeye(3), x_gate, qt.qeye(3), qt.qeye(3), qt.qeye(3)), cnot6, cnot7, hadamard_layer]
    r110 = [hadamard_layer, qt.tensor(x_gate, qt.qeye(3), qt.qeye(3), qt.qeye(3), qt.qeye(3), qt.qeye(3)), qt.tensor(qt.qeye(3), x_gate, qt.qeye(3), qt.qeye(3), qt.qeye(3), qt.qeye(3)), cnot8, cnot9, hadamard_layer]

    recovery_ops = [r000, r001, r010, r011, r100, r101, r110, r111]
    recovery_times = [
        [0.0], 
        [single_qudit_time, single_qudit_time, two_qudit_time, single_qudit_time],
        [single_qudit_time, single_qudit_time, two_qudit_time, single_qudit_time], 
        [single_qudit_time, single_qudit_time, single_qudit_time, two_qudit_time, two_qudit_time, single_qudit_time], 
        [single_qudit_time, single_qudit_time, two_qudit_time, single_qudit_time], 
        [single_qudit_time, single_qudit_time, single_qudit_time, two_qudit_time, two_qudit_time, single_qudit_time], 
        [single_qudit_time, single_qudit_time, single_qudit_time, two_qudit_time, two_qudit_time, single_qudit_time], 
        [0.0]
    ]
    
    return circuit, gate_time, measurements, recovery_ops, recovery_times

In [41]:
def main():

    #Trotterizer Params
    iterations = 2
    t1_list = np.linspace(50, 160, iterations)
    t2_list = np.linspace(33.33, 321, iterations)
    
    trotter_dt = .03
    cnot_time = .5
    single_qudit_time = .03

    

    for i in range(iterations):
        trotterizer = generate_trotterizer(trotter_dt, t1_list[i], t2_list[i], 3, 6)
        experiment_run(trotterizer, single_qudit_time, cnot_time, serial_erasure_circuit, "./output/serial_phase_code")

    
    


In [ ]:
main()